In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
# local mods
import geo_utils as geo
import clean_utils as clean

In [ ]:
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import numpy as np
import joblib
from pathlib import Path

In [ ]:
import random

In [ ]:
# Path to the folder containing spreadsheet files
#path = "spreadsheets/*.csv"
#path = "test1/*.xlsx"
#path = "test3/*.xlsx"
#path = "test5/*.xlsx"
path = "test2026-05-29/*.xlsx"

In [ ]:
files = glob.glob(path)

# Load each file into a list of DataFrames
df_list = [pd.read_excel(file) for file in files]

# Combine them into one master DataFrame and drop na
master_df = pd.concat(df_list, ignore_index=True)
master_df = master_df.dropna(subset=["intersection","text_on_sign_exact"])

In [ ]:
# clean the strings so they have punctuation removed and such
master_df = clean.clean_string_columns(master_df,columns=["text_on_sign_exact"])

In [ ]:
# import additional data for mapping coordinates to intersection names
coordinates = pd.read_csv("coordinate_dict2026-05-29.csv")
platte_river_points = pd.read_csv("platte_points.csv")

In [ ]:
# drop na and transition latitude and longitude strings
# to a float format
coordinates[["latitude", "longitude"]] = (
    coordinates["cd"]
    .dropna()
    .apply(geo.parse_dms_coordinate)
    .apply(pd.Series)
)

In [ ]:
# merge the coordinate sheet with the master sheet
# so the data can be trained
training_data_raw = master_df.merge(
    coordinates[["intersection", "latitude", "longitude", "zip", "city"]],
    on="intersection",
    how="left"
)

In [ ]:
training_data_raw.head(5)

In [ ]:
training_data_raw.to_csv('training_data_raw.csv')

In [ ]:
# create a dataframe that has one row per instersection with
# all of the text combined for that intersection
intersection_df = (
    training_data_raw
    .groupby("intersection")
    .agg({
        "text_on_sign_exact": " ".join,
        "latitude": "first",
        "longitude": "first",
        "zip": "first",
        "city": "first"
    })
    .reset_index()
    .rename(columns={"text_on_sign_exact": "text_blob"})
)

In [ ]:
# remove intersections where geographical data has not been completed
intersection_df = intersection_df.dropna(subset=["latitude","longitude"])
# create a checkpoint dataset
intersection_df.to_csv("output/cleaned_concatenated.csv")

In [ ]:
coords = intersection_df[["latitude", "longitude"]]

# perform k-means clustering on the coordinates
# for optional classification algorithm later on
kmeans = KMeans(n_clusters=4, random_state=42, n_init="auto")

intersection_df["spatial_cluster"] = kmeans.fit_predict(coords)

In [ ]:
intersection_df

In [ ]:
#intersection_df = intersection_df[
#    intersection_df["city"].isin(["denver", "englewood", "cherry creek", "aurora"])
#]

In [ ]:
intersection_df

In [ ]:
plt.figure(figsize=(12,8))

plt.text(-104.996583333333, 39.74852777777778,"Downtown Denver")
plt.text(-104.875277777777, 39.74022222222222, "Aurora on Colfax")
plt.text(-104.768055555555, 39.53711111111111, "Parker/Lincoln")

plt.scatter(
    intersection_df["longitude"], 
    intersection_df["latitude"], 
    color = "orange", 
    label = "Trained Intersections")
plt.show()

# Predicting a Single Coordinate Based on Language Data from an Intersection
## Distance

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate great-circle distance between two points on Earth.
    
    Returns distance in kilometers.
    """

    R = 6371  # Earth radius in km

    # convert degrees to radians
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)

    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    # differences
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # haversine formula
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [ ]:
# set up dataset of optional data training process
training_data = training_data_raw.dropna(subset=["latitude","longitude"])

In [ ]:
# create function that will create a sample training dataset
# based on random combinations of linguistic landscape observations
def generate_samples(signs, n_samples=50,
                     min_size=3,
                     max_size=7,
                     seed=None):

    rng = random.Random(seed)
    samples = []

    for _ in range(n_samples):

        lower = min(min_size, len(signs))
        upper = min(max_size, len(signs))

        size = rng.randint(lower, upper)
        subset = rng.sample(signs, size)

        samples.append(" ".join(subset))

    return samples

In [ ]:
# create random sample with the training data
augmented_rows = []
seed = 42

for intersection, group in training_data.groupby("intersection"):
    
    signs = group["text_on_sign_exact"].dropna().tolist()
    
    lat = group["latitude"].iloc[0]
    lon = group["longitude"].iloc[0]
    
    samples = generate_samples(
        signs,
        n_samples=50,
        min_size=5,
        max_size=10,
        seed=seed
    )
    seed += 1
    
    for sample in samples:
        augmented_rows.append({
            "intersection": intersection,
            "sample_text": sample,
            "lat": lat,
            "lon": lon
        })

In [ ]:
test = pd.DataFrame(augmented_rows)
test

In [ ]:
# assign variables based on original dataset
X = intersection_df["text_blob"]

# for using geographical coordinates
y = intersection_df[["latitude", "longitude"]]

# for using kmean clusters
# y = intersection_df["spatial_cluster"]   # or "city", "region_label", etc.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state= 348#24#2 # 55 #46 # 42
)

In [ ]:
coord_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        lowercase=True,
        min_df=2
    )),
    ("regressor", MultiOutputRegressor(
        Ridge(alpha=0.1)
    ))
])

In [ ]:
X_test

In [ ]:
# fit the model
# model.fit(X_train, y_train)
coord_model.fit(X_train, y_train)

In [ ]:
Path("models").mkdir(exist_ok=True)

coord_model.fit(X_train, y_train)

joblib.dump(coord_model, "models/coord_model.joblib")

In [ ]:
# test out the training data
pred_coords = coord_model.predict(X_test)

pred_df = y_test.copy()
pred_df["pred_latitude"] = pred_coords[:, 0]
pred_df["pred_longitude"] = pred_coords[:, 1]

pred_df.head()

In [ ]:
# assigning mins and maxs to set bounds for random points
lat_min = min(pred_df["latitude"].min(), pred_df["pred_latitude"].min())
lat_max = max(pred_df["latitude"].max(), pred_df["pred_latitude"].min())
long_min = min(pred_df["longitude"].min(),pred_df["pred_longitude"].min())
long_max = max(pred_df["longitude"].max(), pred_df["longitude"].max())

In [ ]:
# create random points for latitude and longitude
np.random.seed(28)

pred_df["random_latitude"] = np.random.uniform(
    lat_min,
    lat_max,
    size=len(pred_df)
)

pred_df["random_longitude"] = np.random.uniform(
    long_min,
    long_max,
    size=len(pred_df)
)

In [ ]:
# append random error to spreadsheet
pred_df["random_error_km"] = haversine_distance(
    pred_df["latitude"],
    pred_df["longitude"],
    pred_df["random_latitude"],
    pred_df["random_longitude"]
)

In [ ]:
mae_lat = mean_absolute_error(y_test["latitude"], pred_df["pred_latitude"])
mae_lng = mean_absolute_error(y_test["longitude"], pred_df["pred_longitude"])

print("Latitude MAE:", mae_lat)
print("Longitude MAE:", mae_lng)

In [ ]:
pred_df["error_km"] = haversine_distance(
    pred_df["latitude"],
    pred_df["longitude"],
    pred_df["pred_latitude"],
    pred_df["pred_longitude"]
)

pred_df[[
    "latitude",
    "longitude",
    "pred_latitude",
    "pred_longitude",
    "error_km"
]].head(7)

In [ ]:
print("Mean error in km:", pred_df["error_km"].mean())
print("Median error in km:", pred_df["error_km"].median())

In [ ]:
plt.figure(figsize=(12,8))

plt.text(-104.996583333333, 39.74852777777778,"Downtown Denver")
plt.text(-104.875277777777, 39.74022222222222, "Aurora on Colfax")
plt.text(-104.768055555555, 39.53711111111111, "Parker/Lincoln")


# true locations
plt.scatter(
    pred_df["longitude"],
    pred_df["latitude"],
    color="blue",
    label="True",
    s=80
)

# predicted locations
plt.scatter(
    pred_df["pred_longitude"],
    pred_df["pred_latitude"],
    color="red",
    label="Predicted",
    s=80
)

# arrows / lines
for _, row in pred_df.iterrows():

    plt.plot(
        [row["longitude"], row["pred_longitude"]],
        [row["latitude"], row["pred_latitude"]],
        color="gray",
        alpha=0.5
    )



plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.title("Prediction Drift, 2000+ Rows")

plt.legend()

plt.savefig("prediction_drift2000_rows.png")

plt.show()

In [ ]:
plt.figure(figsize=(12,8))

plt.text(
    -104.996583333333,
    39.74852777777778,
    "Downtown Denver"
)

plt.text(
    -104.875277777777,
    39.74022222222222,
    "Aurora on Colfax"
)
plt.text(
    -104.768055555555,
    39.53711111111111, 
    "Parker/Lincoln")

# TRUE LOCATIONS
plt.scatter(
    pred_df["longitude"],
    pred_df["latitude"],
    color="blue",
    label="True",
    s=80
)

# MODEL PREDICTIONS
plt.scatter(
    pred_df["pred_longitude"],
    pred_df["pred_latitude"],
    color="red",
    label="Model Prediction",
    s=80
)

# RANDOM PREDICTIONS
plt.scatter(
    pred_df["random_longitude"],
    pred_df["random_latitude"],
    color="green",
    label="Random Prediction",
    s=80,
    alpha=0.7
)

# MODEL LINES + ERROR LABELS
for _, row in pred_df.iterrows():

    # model prediction line
    plt.plot(
        [row["longitude"], row["pred_longitude"]],
        [row["latitude"], row["pred_latitude"]],
        color="gray",
        alpha=0.5
    )

    # midpoint for annotation
    mid_x = (
        row["longitude"] +
        row["pred_longitude"]
    ) / 2

    mid_y = (
        row["latitude"] +
        row["pred_latitude"]
    ) / 2

    # model error label
    plt.text(
        mid_x,
        mid_y,
        f"{row['error_km']:.1f} km",
        fontsize=7,
        alpha=0.7,
        color="black"
    )

# RANDOM LINES
for _, row in pred_df.iterrows():

    plt.plot(
        [row["longitude"], row["random_longitude"]],
        [row["latitude"], row["random_latitude"]],
        color="green",
        alpha=0.25,
        linestyle="dashed"
    )

plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.title("Model vs Random Geographic Prediction Drift")

plt.legend()

plt.savefig("test_random_vs_model.png", dpi=300)

plt.show()

In [ ]:
print("MODEL")
print(pred_df["error_km"].describe())

print("\nRANDOM")
print(pred_df["random_error_km"].describe())

In [ ]:
query = "taqueria mercado pho celulares"

predicted_coords = coord_model.predict([query])

pred_lat = predicted_coords[0][0]
pred_lon = predicted_coords[0][1]

print(pred_lat, pred_lon)

# Predict Possible Areas Based on Intersection Text Data
# using "Neighbors

In [ ]:
# grab the vectorizer to visualize
# vectorizer = model.named_steps["tfidf"]
vectorizer = coord_model.named_steps["tfidf"]

X_all_tfidf = vectorizer.transform(intersection_df["text_blob"])

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn_model = NearestNeighbors(
    n_neighbors=5,
    metric="cosine"
)

nn_model.fit(X_all_tfidf)

In [ ]:
def find_linguistic_neighbors(text, intersection_df, vectorizer, nn_model, n_neighbors=5):
    text_vec = vectorizer.transform([text])

    distances, indices = nn_model.kneighbors(
        text_vec,
        n_neighbors=n_neighbors
    )

    results = intersection_df.iloc[indices[0]].copy()
    results["cosine_distance"] = distances[0]
    results["similarity"] = 1 - results["cosine_distance"]

    return results[
        [
            "intersection",
            "city",
            "latitude",
            "longitude",
            "similarity",
            "text_blob"
        ]
    ]

In [ ]:
def plot_knn_triangulation(knn_df, n, actual:tuple):
    knn_df = knn_df[:n]
    average = triangulate_mean(knn_df, n)
    plt.scatter(knn_df["longitude"], knn_df["latitude"],
           label = f"Top {n} Projected Points")
    plt.scatter(actual[0], actual[1], label = "Actual Intersection")
    plt.scatter(average[0], average[1], label = "Triangulation")
    plt.legend()
    plt.show()

In [ ]:
def triangulate_mean(knn_df, n):
    knn_df = knn_df[:n]
    knn_longitude_average = knn_df["longitude"].mean()
    knn_latitude_average = knn_df["latitude"].mean()
    return((knn_longitude_average, knn_latitude_average))

In [ ]:
# test a query on Grant
query_text = """
taco river autozone pho vietnamnese restaurant crawfish market pho
"""
query_point = (-105.025207, 39.694929) 

test_knn = find_linguistic_neighbors(
    query_text,
    intersection_df,
    vectorizer,
    nn_model,
    n_neighbors=5
)

In [ ]:
test_knn

In [ ]:
plot_knn_triangulation(test_knn, n= 2, actual = query_point)

In [ ]:
# take the top four closest in linguistic space
test_knn = test_knn[:3]

knn_longitude_average = test_knn["longitude"].mean()
knn_latitude_average = test_knn["latitude"].mean()

In [ ]:
plt.scatter(test_knn["longitude"], test_knn["latitude"],
           label = "Top Three Projected Points")
plt.scatter(query_point[0], query_point[1], label = "Actual Intersection")
plt.scatter(knn_longitude_average, knn_latitude_average, label = "Prediction")
plt.legend()
plt.show()

In [ ]:
# visualize the gram weights
X_test_tfidf = vectorizer.transform(X_test)
feature_names = vectorizer.get_feature_names_out()

# model for kmean space
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        lowercase=True,
        min_df=2
    )),
    ("clf", LogisticRegression(
        max_iter=5000
    ))
])

# for kmean clusters
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
feature_names

In [ ]:
# test looking at one row
row = X_test_tfidf[1]
weights = row.toarray()[0]

In [ ]:
tfidf_scores = list(zip(feature_names, weights))

In [ ]:
X_test

In [ ]:
preds = model.predict(X_test)

print(classification_report(y_test, preds))
print(confusion_matrix(y_test, preds))
print(type(preds))

In [ ]:
plt.figure(figsize=(10, 8))

plt.scatter(
    intersection_df["longitude"],   # x-axis
    intersection_df["latitude"],    # y-axis
    c=intersection_df["spatial_cluster"]
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Spatial Clusters by Intersection")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.scatter(coordinates["longitude"], coordinates["latitude"], color="orange")
ax.plot(platte_river_points["platte_long"], platte_river_points["platte_lat"])
ax.set_xlabel("Longitude", weight = "semibold")
ax.set_ylabel("Latitude", weight="semibold")
ax.set_title("Intersections in the Greater Denver Metro Area", weight="bold")
fig.show()